In [3]:
"""
barrier_strain_sensitivity.py
=============================

Strain/stress sensitivity of an NEB barrier from a MACE potential, via autograd.

Workflow this supports
----------------------
  1. You converge a climbing-image NEB in ASE with MACECalculator (your existing machinery).
  2. You hand three ASE Atoms objects to `barrier_sensitivities()`:
         initial minimum, climbing image (saddle), final minimum
  3. You get back, per output, a 3x3 tensor of d(energy difference)/d(strain):
         A_fwd = forward barrier sensitivity
         A_rxn = reaction energy sensitivity
         A_rev = reverse barrier sensitivity
 
Why this works without differentiating through the NEB
------------------------------------------------------
All three geometries are stationary points, so dE*/deta = partial E/partial eta with the
geometry held fixed (envelope theorem). Each configuration therefore contributes exactly one
tensor -- its virial W = dE/deta = V*sigma -- and every output is a difference of virials.
 
The strain enters through MACE's `displacement` hook: `prepare_graph()` reads
`data.get("displacement")` and, when compute_stress=True, applies
 
    positions <- positions + positions @ sym(eta)
    cell      <- cell      + cell      @ sym(eta)
    shifts    <- unit_shifts @ cell_strained
 
so `eta` is a genuine leaf of the autograd graph.
 
Requires: mace-torch, ase, torch.  CPU is fine for a single evaluation.
"""
 
from __future__ import annotations
 
import numpy as np
import torch
 
# ---------------------------------------------------------------------------------------
# CRITICAL: set the default dtype BEFORE constructing any AtomicData.
# AtomicData.from_config() calls torch.get_default_dtype() internally. If you set this after
# building the batch you get float32 tensors fed to a float64 model, and the resulting
# gradients are garbage at the 1e-4 level -- which is exactly the size of the effect you are
# trying to measure.
# ---------------------------------------------------------------------------------------
torch.set_default_dtype(torch.float64)
 
from mace.data import AtomicData, config_from_atoms  # noqa: E402
from mace.tools import AtomicNumberTable  # noqa: E402
from mace.tools.torch_geometric.batch import Batch  # noqa: E402
 
 
# =======================================================================================
# 1. Model loading
# =======================================================================================

def load_mace(model_or_path, device="cpu", dtype=torch.float64, freeze=True):
    """Load a raw MACE module (NOT MACECalculator, NOT MaceTorchSimModel).
 
    Do not enable cuEquivariance / OpenEquivariance / torch.compile on this instance.
    Those fused kernels are opaque to AOTAutograd and are not guaranteed to support the
    double-backward you need for Hessians and mixed second derivatives.
    """
    if isinstance(model_or_path, torch.nn.Module):
        model = model_or_path
    else:
        model = torch.load(model_or_path, map_location=device, weights_only=False)
 
    model = model.to(device=device, dtype=dtype).eval()
    if freeze:
        # Set freeze=False if you ever want dE/d(weights) -- barrier sensitivity to the
        # model parameters, e.g. for committee uncertainty propagation.
        for p in model.parameters():
            p.requires_grad_(False)
    return model
 
 
# =======================================================================================
# 2. ASE -> MACE translation
# =======================================================================================
 
class MaceBatcher:
    """Turns a list of ASE Atoms into a single batched MACE input dict.
 
    Batching all images into ONE forward pass is the main performance win over looping
    the ASE calculator per image.
    """
 
    def __init__(self, model, head: str | None = None):
        self.z_table = AtomicNumberTable([int(z) for z in model.atomic_numbers])
        self.r_max = float(model.r_max)
        self.heads = list(getattr(model, "heads", ["Default"]))
        self.head = head if head is not None else self.heads[0]
        self.device = next(model.parameters()).device
        self.dtype = next(model.parameters()).dtype
 
    def build(self, atoms_list) -> dict:
        """Returns a plain dict ready for model(...).
 
        NOTE the neighbour list is built here, in numpy, from the *current* positions and
        is not differentiable. That is correct and is the same assumption behind any
        analytic stress: the derivative is exact for a fixed edge set. It is only wrong if
        a finite strain step pushes a pair across r_max, which does not happen for the
        infinitesimal derivative we take here.
        """
        datas = []
        for atoms in atoms_list:
            cfg = config_from_atoms(atoms, head_name=self.head)
            datas.append(
                AtomicData.from_config(
                    cfg, z_table=self.z_table, cutoff=self.r_max, heads=self.heads
                )
            )
        batch = Batch.from_data_list(datas).to(self.device)
        return batch.to_dict()
 
 
# =======================================================================================
# 3. The autograd hooks
# =======================================================================================
 
def evaluate(model, batch_dict, strain=None, atomic=False):
    """One batched forward pass with the graph kept alive.
 
    training=True is MANDATORY, and not for the reason you might assume. MACE computes
    forces/virials internally via torch.autograd.grad(..., retain_graph=training). With
    training=False the graph from energy back to positions and displacement is FREED by
    MACE's own call, and any subsequent grad() of yours raises
    "Trying to backward through the graph a second time".
    """
    # prepare_graph() WRITES BACK into the dict you hand it
    #     data["positions"], data["shifts"] = p, s
    # so reusing a dict double-applies the strain. Always pass a fresh shallow copy.
    d = dict(batch_dict)
    if strain is not None:
        d["displacement"] = strain
 
    return model(
        d,
        training=True,
        compute_force=True,
        compute_stress=True,          # required, or the displacement hook is ignored
        compute_edge_forces=atomic,   # needed to get atomic virials
        compute_atomic_stresses=atomic,
    )
 
 
def energies_and_virials(model, batch_dict, n_systems, atomic=False):
    """Returns (E [n], W [n,3,3], out).
 
    W[x] = dE_x/deta, the virial of configuration x. Every downstream quantity is a
    difference of these, so compute them once.
    """
    strain = torch.zeros(
        n_systems, 3, 3,
        dtype=batch_dict["positions"].dtype,
        device=batch_dict["positions"].device,
        requires_grad=True,
    )
    out = evaluate(model, batch_dict, strain=strain, atomic=atomic)
    E = out["energy"]                                   # [n_systems]
 
    # A single backward on E.sum() yields all n virials with no cross-contamination,
    # because each E_x depends only on its own 3x3 slice of `strain`.
    (W,) = torch.autograd.grad(E.sum(), strain, retain_graph=True)
    return E.detach(), W.detach(), out, strain
 
 
# =======================================================================================
# 4. Main entry point
# =======================================================================================
 
def barrier_sensitivities(
    model, batcher, atoms_initial, atoms_saddle, atoms_final,
    atomic=True, fmax_tol=2e-3, verbose=True,
):
    """Three converged geometries in, two (plus one dependent) sensitivity tensors out.
 
    Returns a dict. All gradients are in eV per UNIT strain -- divide by 100 for
    eV per 1% strain, which is the number worth quoting.
    """
    _assert_consistent(atoms_initial, atoms_saddle, atoms_final)
 
    batch = batcher.build([atoms_initial, atoms_saddle, atoms_final])
    E, W, out, _ = energies_and_virials(model, batch, n_systems=3, atomic=atomic)
 
    # ---- stationarity gate -------------------------------------------------------------
    # The envelope theorem needs dE/dR = 0 at all three points. The dropped term scales as
    # |F_resid| * ||dR*/deta||, so under-converged geometries silently bias the answer.
    # Recompute fmax HERE rather than trusting the NEB's number: if you converged with a
    # different neighbour-list backend, the residual force on this PES may not be what ASE
    # reported. Note these are RAW forces -- ASE zeroes constrained components.
    ptr = batch["ptr"]
    F = out["forces"].detach()
    fmax = [F[ptr[k]:ptr[k + 1]].norm(dim=1).max().item() for k in range(3)]
    labels = ("initial", "saddle", "final")
    for lbl, fm in zip(labels, fmax):
        if fm > fmax_tol:
            print(f"  WARNING: {lbl} fmax = {fm:.2e} eV/A > {fmax_tol:.0e}. "
                  f"Envelope theorem error ~ this magnitude. Converge harder.")
 
    # ---- the outputs -------------------------------------------------------------------
    A_fwd = W[1] - W[0]     # forward barrier   E_s - E_i
    A_rxn = W[2] - W[0]     # reaction energy   E_f - E_i
    A_rev = W[1] - W[2]     # reverse barrier   E_s - E_f
 
    # ---- consistency checks ------------------------------------------------------------
    # (a) The hook symmetrises, so every W must be symmetric. If not, the strain never
    #     reached the model and you are differentiating a zero tensor.
    for k, lbl in enumerate(labels):
        assert torch.allclose(W[k], W[k].T, atol=1e-9), \
            f"W[{lbl}] not symmetric -- displacement hook did not take effect"
    assert W.abs().max() > 0, "all virials are zero -- check compute_stress=True"
 
    # (b) W must equal V*sigma. This is an independent code path inside MACE, so it is a
    #     genuine cross-check of your autograd plumbing.
    #     Holds for slabs too: MACE defines stress = virial/V, so V*sigma reconstructs the
    #     virial exactly and the vacuum padding cancels. Only sigma ALONE is vacuum-diluted.
    V = torch.linalg.det(batch["cell"].view(-1, 3, 3)).abs()
    W_from_stress = V.view(-1, 1, 1) * out["stress"].detach()
    assert torch.allclose(W, W_from_stress, atol=1e-8), \
        f"W != V*sigma (max dev {(W - W_from_stress).abs().max():.2e})"
 
    # (c) Algebraically vacuous (all three come from the same W), but catches index and
    #     sign typos for free. This is NOT validation -- see the tier-3 FD check.
    assert torch.allclose(A_fwd - A_rev, A_rxn, atol=1e-12)
 
    result = dict(
        E_initial=E[0].item(), E_saddle=E[1].item(), E_final=E[2].item(),
        barrier_fwd=(E[1] - E[0]).item(),
        barrier_rev=(E[1] - E[2]).item(),
        E_rxn=(E[2] - E[0]).item(),        # 0 K internal energy. NOT a free energy.
        W_initial=W[0], W_saddle=W[1], W_final=W[2],
        A_fwd=A_fwd, A_rxn=A_rxn, A_rev=A_rev,
        A_fwd_voigt=to_voigt(A_fwd), A_rxn_voigt=to_voigt(A_rxn),
        fmax=dict(zip(labels, fmax)),
        volume=V.detach(),
    )
 
    # ---- per-atom decomposition --------------------------------------------------------
    if atomic:
        # MACE returns atomic_virials with the opposite sign to dE/deta (it follows the
        # `virials = -dE/deta` convention). Flip so it sums to +W like everything else.
        w_atom = -out["atomic_virials"].detach()
        sums = torch.stack([w_atom[ptr[k]:ptr[k + 1]].sum(0) for k in range(3)])
        dev = (sums - W).abs().max()
        assert dev < 1e-6, (
            f"atomic virial sum rule violated ({dev:.2e}). If it is off by exactly a sign, "
            f"drop the leading minus above -- MACE's virial sign conventions differ between "
            f"`virials` and the raw gradient."
        )
        n = len(atoms_initial)
        result["dw_fwd"] = w_atom[ptr[1]:ptr[2]] - w_atom[ptr[0]:ptr[1]]
        result["dw_rxn"] = w_atom[ptr[2]:ptr[3]] - w_atom[ptr[0]:ptr[1]]
        assert result["dw_fwd"].shape == (n, 3, 3)
        # Caveat: the atomic partition is gauge-dependent -- only the total is unique.
        # Differences between configurations of the same system are more robust than
        # absolute per-atom values. Use for spatial localisation, not for absolute claims.
 
    if verbose:
        _report(result)
    return result
 
 
# =======================================================================================
# 5. Reporting helpers
# =======================================================================================
 
def to_voigt(A):
    """3x3 gradient -> Voigt 6, conjugate to ENGINEERING strain.
 
    PLAIN COPY, no factor of 2. The energy differential is a full double contraction, so
    each off-diagonal pair contributes 2*A_xy*d(eta_xy); the engineering definition
    gamma_xy = 2*eta_xy absorbs exactly that factor. A converts like a stress, because
    A = V*sigma. It is the STRAIN vector that gets the 2, not the gradient.
    """
    return torch.stack([A[0, 0], A[1, 1], A[2, 2], A[1, 2], A[0, 2], A[0, 1]])
 
 
def sensitivity_along(A, direction):
    """Sensitivity to one loading mode, in eV per unit strain. Convention-free.
 
    Examples:
        uniaxial along n :  torch.outer(n, n)
        shear on (n, m)  :  0.5 * (torch.outer(n, m) + torch.outer(m, n))
        hydrostatic      :  torch.eye(3)
    """
    d = 0.5 * (direction + direction.T)
    d = d / d.norm()
    return (A * d).sum()
 
 
def _report(r):
    print(f"\n  E_initial {r['E_initial']:14.6f} eV")
    print(f"  E_saddle  {r['E_saddle']:14.6f} eV")
    print(f"  E_final   {r['E_final']:14.6f} eV")
    print(f"\n  barrier (fwd) {r['barrier_fwd']:10.4f} eV")
    print(f"  barrier (rev) {r['barrier_rev']:10.4f} eV")
    print(f"  E_rxn         {r['E_rxn']:10.4f} eV   [0 K internal energy, no entropy/ZPE]")
    names = ["xx", "yy", "zz", "yz", "xz", "xy"]
    print("\n  sensitivity, eV per 1% strain (Voigt, engineering shear):")
    print(f"  {'':>6}" + "".join(f"{nm:>10}" for nm in names))
    for key, lbl in (("A_fwd_voigt", "barrier"), ("A_rxn_voigt", "E_rxn")):
        vals = (r[key] / 100.0).tolist()
        print(f"  {lbl:>6}" + "".join(f"{v:10.4f}" for v in vals))
    print(f"\n  fmax: " + "  ".join(f"{k}={v:.1e}" for k, v in r["fmax"].items()))
 
 
def _assert_consistent(*atoms_list):
    ref = atoms_list[0]
    for i, a in enumerate(atoms_list[1:], 1):
        assert len(a) == len(ref), f"image {i}: atom count differs"
        assert (a.get_atomic_numbers() == ref.get_atomic_numbers()).all(), \
            f"image {i}: atom ORDER differs -- per-atom decomposition would be meaningless"
        assert np.allclose(a.get_cell(), ref.get_cell(), atol=1e-8), \
            f"image {i}: cell differs -- this code assumes fixed-cell (strain-controlled) NEB"
 
 
# =======================================================================================
# 6. Validation: finite-difference the virial through an independent path
# =======================================================================================
 
def fd_check_virial(model, batcher, atoms, direction=None, h=1e-4):
    """Validate the displacement hook end-to-end against a real deformation.
 
    This does NOT test the envelope theorem (for that you must re-converge NEBs at
    eta +/- h). It DOES test that the hook, the symmetrisation, the cell/shift update and
    your sign conventions are all correct -- which is the part most likely to be wrong,
    and it costs three single-point energies.
    """
    if direction is None:
        direction = torch.tensor([[0.0, 0.5, 0.0], [0.5, 0.0, 0.0], [0.0, 0.0, 0.0]])
    D = 0.5 * (direction + direction.T)
 
    # analytic
    batch = batcher.build([atoms])
    _, W, _, _ = energies_and_virials(model, batch, n_systems=1)
    analytic = (W[0] * D).sum().item()
 
    # numeric: affine deformation of cell + positions, exactly what the hook emulates
    I = np.eye(3)
    Dn = D.numpy()
    def energy_at(eps):
        a = atoms.copy()
        a.set_cell(np.array(atoms.get_cell()) @ (I + eps * Dn), scale_atoms=True)
        b = batcher.build([a])
        with torch.no_grad():
            e = model(dict(b), training=False, compute_force=False,
                      compute_stress=False)["energy"]
        return e[0].item()
 
    numeric = (energy_at(h) - energy_at(-h)) / (2 * h)
    rel = abs(analytic - numeric) / max(abs(numeric), 1e-12)
    print(f"  FD check: analytic {analytic:.8f}  numeric {numeric:.8f}  rel err {rel:.2e}")
    assert rel < 1e-5, "displacement hook does not match a real affine deformation"
    return analytic, numeric
 
 
def n_negative_eigenvalues(model, batcher, atoms):
    """Hessian eigenvalue count. MUST be 1 at the saddle and 0 at the endpoints.
 
    Cheapest reliable correctness alarm you have. Cost is O(3N) backward passes, so this is
    for modest cells; skip or subsample for large ones.
    """
    batch = batcher.build([atoms])
    d = dict(batch)
    out = model(d, training=True, compute_force=True, compute_stress=False,
                compute_hessian=True)
    n = len(atoms)
    H = out["hessian"].reshape(3 * n, 3 * n).detach()
    H = 0.5 * (H + H.T)
    evals = torch.linalg.eigvalsh(H)
    # Translational zero modes sit near 0; use a tolerance well below real mode curvatures.
    return int((evals < -1e-4).sum().item()), evals[:8]

/home/ysx6266/.conda/envs/atomistic/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


In [2]:
# ------------------------------- #
# Imports
    # System
import os
import sys
import json
import shutil as sh
import glob
import re

    # System +
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

    # File io
from ase.io import read, write

    # Other
from ase.visualize.plot import plot_atoms
from ase import Atoms

    # Constraints
from ase.constraints import FixAtoms

    # Defect simulation tools
from ase.build import make_supercell, find_optimal_cell_shape

# Mine
from DFTUtils import write_vasp_settings, write_settings_json, copy_files_from_DFTUtilities, remove_files, make_directories_from_list
from DFTUtils import get_strain, get_interstitials, aggregate_unique_structures
from dftjobs import DFTJobManager

# ---------------------------- #
# Globals
H_CHEMPOT = -0.6455593929751439 # -3.4633222742407046 <- for a molecule, but we are doing radicals, -0.6455593929751439 <- radical

# ---------------------------- #
notebook_path = os.path.dirname(__vsc_ipynb_file__)

In [ ]:
os.chdir(notebook_path)
os.chdir('/projects/p32212/Collaborator_Projects/Duncan_Sn/H_Diffusion/[100]')

job = DFTJobManager('quest', walltime = '00:08:00', clear_old_logs = False)
job.submit_script('MACE_Barrier_Sensitivity.py', dry_run = False)

b'Submitted batch job 9305590'


9305590

In [1]:
# =======================================================================================
# 7. Example
# =======================================================================================

# --- your converged CI-NEB -----------------------------------------------------------
# images = neb.images                      # from your existing machinery
images = read("NEB.traj@-7:")
energies = np.array([img.get_potential_energy() for img in images])
ci = int(energies.argmax())
assert 0 < ci < len(images) - 1, "highest image is an endpoint -- band not converged"
print(f"climbing image = {ci} of {len(images) - 1}")

atoms_i, atoms_s, atoms_f = images[0], images[ci], images[-1]

# --- model --------------------------------------------------------------------------- #
from mace.calculators import mace_mp
calc = mace_mp('mh-1', head = 'oc20_usemppbe')
# model = load_mace(notebook_path + "/../../Interatomic_Potentials/mace-mh-1(2).model", device="cpu")
model = load_mace(calc.models[0], device="cpu")
batcher = MaceBatcher(model)

# --- validate before trusting anything ------------------------------------------------
fd_check_virial(model, batcher, atoms_i)

for lbl, a, expect in (("initial", atoms_i, 0), ("saddle", atoms_s, 1),
                        ("final", atoms_f, 0)):
    n_neg, lo = n_negative_eigenvalues(model, batcher, a)
    print(f"  {lbl:>7}: {n_neg} negative eigenvalue(s), expected {expect}")
    assert n_neg == expect, f"{lbl} is not a proper stationary point"

# --- the actual result ----------------------------------------------------------------
r = barrier_sensitivities(model, batcher, atoms_i, atoms_s, atoms_f)

# --- a specific loading mode ----------------------------------------------------------
n_dir = torch.tensor([1.0, 1.0, 0.0])
uniaxial_110 = torch.outer(n_dir, n_dir)
s = sensitivity_along(r["A_fwd"], uniaxial_110) / 100.0
print(f"\n  d(barrier)/d(uniaxial [110]) = {s:+.4f} eV per 1% strain")

# --- where is the response localised? -------------------------------------------------
if "dw_fwd" in r:
    mag = r["dw_fwd"].flatten(1).norm(dim=1)
    top = mag.argsort(descending=True)[:5]
    print("\n  atoms carrying the most barrier stress-response:")
    for a_idx in top.tolist():
        print(f"    atom {a_idx:4d}  {atoms_i.symbols[a_idx]:>2}  |dw| = {mag[a_idx]:.4f} eV")

NameError: name 'read' is not defined